In [ ]:
import csv
import numpy as np
from collections import defaultdict

try:
    from gurobipy import Model, GRB
except Exception as e:
    raise ImportError("gurobipy not found. Install gurobipy and ensure you have a license.") from e

# Function Definitions

In [ ]:
def is_parallel(a, b,eps):
    # under the assumption that both vectors are normalized...
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if len(a) != len(b):
        print("WARNING: Tried to check whether vectors are parallel, but they do not have the same dimension.")
    if len(a) ==3:
        if abs(a[0]*b[0]+a[1]*b[1]+a[2]*b[2])>1-eps:
            return True
        else:
            return False
    elif len(a) ==2:
        if abs(a[0]*b[0]+a[1]*b[1])>1-eps:
            return True
        else:
            return False
    else:
        print("WARNING: Tried to check whether vectors are parallel, but they are neither 2D nor 3D vectors.")


def canonicalize_undirected(vec):
    # make vec's sign deterministic (so vec and -vec become the same canonical direction).
    # Rule: first non-zero component must be positive.
    vec = np.asarray(vec, dtype=float)
    for c in vec:
        if abs(c) > 1e-12:
            return vec if c > 0 else -vec
    return vec 

def compute_plane_normals(vecs,eps):
    # Given unit vectors, return list of (i,j,n_ij) for pairs where ||v_i x v_j|| > eps. n_ij is unit normal.

    unique_normals=[]
    m = vecs.shape[0]
    pairs = []
    for i in range(m):
        for j in range(i+1, m):
            check_parallel=vecs[i][0]*vecs[j][0]+vecs[i][1]*vecs[j][1]+vecs[i][2]*vecs[j][2]
            if abs(check_parallel) < 1-eps:
                cross = np.cross(vecs[i], vecs[j])
                norm = np.linalg.norm(cross)
                if norm > eps:
                    new=cross / norm;
                    DoAdd=True
                    for v in unique_normals:
                        if is_parallel(v, new,eps):
                            DoAdd=False
                    if DoAdd:
                        new=np.round(new, 6)
                        unique_normals.append(new)
                        pairs.append((i, j, new))
    print("I found",len(unique_normals),"planes")
    return pairs

def normalize_vec(a):
    # normalize vector
    a = np.asarray(a, dtype=float)
    return a / np.linalg.norm(a)

def read_csv(file, dim, SD,eps):
    nodes = set()
    if SD:
        all_vectors = []
    else:
        vectors_by_node = defaultdict(list)  # node -> list of canoncal vectors adjacent to node
        normals_by_node = defaultdict(list)  # node -> list of canoncal vectors adjacent to node

    with open(file, newline='') as csvfile:
        reader = csv.DictReader(csvfile, delimiter=';')

        for row in reader:
            s = int(row['s'])
            t = int(row['t'])
            nodes.update([s, t])
            if dim == 3:
                vec = np.array([
                    float(row['x2']) - float(row['x1']),
                    float(row['y2']) - float(row['y1']),
                    float(row['z2']) - float(row['z1']),
                ], dtype=float)
            else:
                vec = np.array([
                    float(row['x2']) - float(row['x1']),
                    float(row['y2']) - float(row['y1']),
                ], dtype=float)
            vec = normalize_vec(vec) 
            vec_c = canonicalize_undirected(vec)

            if SD:
                do_add = True
                for v in all_vectors:
                    if is_parallel(v, vec_c,eps):
                        do_add = False
                        break
                if do_add:
                    all_vectors.append(vec_c)
            else:
                # edge is adjacent to BOTH nodes
                vectors_by_node[s].append(vec_c)
                vectors_by_node[t].append(vec_c)
    nodes = sorted(nodes)

    if SD:
        unique_beams = np.asarray(all_vectors, dtype=float)
        print("I found", unique_beams.shape[0], "beams")
        if dim==3:
            normals = compute_plane_normals(unique_beams,eps)
        else:
            normals=[]
        return unique_beams, normals, nodes
    else:
        for n in nodes:
            print("For node ", n, ":")
            vectors_by_node[n] = vectors_by_node.get(n, [])
            print("I found", np.asarray(vectors_by_node[n], dtype=float).shape[0], "beams")
            if dim==3:
                normals_by_node[n] = compute_plane_normals(np.asarray(vectors_by_node[n], dtype=float),eps)
            else:
                normals=[]
        return dict(vectors_by_node),dict(normals_by_node), nodes
    
def sample_warm_start(vecs, normals, n_samples=2000):
    #sample unit vectors x and and return best (x, c) pair found (minimizing c).
    best = None
    m = vecs.shape[0]
    for _ in range(n_samples):
        v = np.random.normal(size=dimension)
        v /= np.linalg.norm(v)
        # beam part
        beam_max = np.max(np.abs(vecs @ v)) if m>0 else 0.0
        # plane part
        plane_req = 0.0
        for (_, _, n) in normals:
            val = 1.0 - (float(n @ v))**2
            if val > 0:
                plane_req = max(plane_req, np.sqrt(val))
        c_req = max(beam_max, plane_req)
        if c_req <= 1.0:
            if (best is None) or (c_req < best[1]):
                best = (v.copy(), float(c_req))
    return best

def solve_disturbance_direction(dimension,vecs,normals,tau,time_limit,warmstart_samples,verbose):
    """
    Build and solve Model (1) from the paper:
      minimize c
      s.t.  +/- x^T u_e <= c,  for all beams e
            (x^T n_ef )^2 + c^2 >= 1, for all plane normals n_ef
            x^T x == 1
            0 <= c <= 1
    Inputs:
      vecs : beam direction vectors       
      tau : tolerance to skip nearly-parallel beam pairs when forming plane normals
      time_limit : seconds for Gurobi
      warmstart_samples : number of random samples to find a feasible warm start (0 to skip)
      verbose: Boolean; turns off/on Gurobi output
    """
    m_beams = vecs.shape[0]


    # Create model
    model = Model("disturbance_direction")
    model.setParam("OutputFlag", 1 if verbose else 0)
    model.setParam("NonConvex", 2)
    if time_limit is not None:
        model.setParam("TimeLimit", time_limit)

    # Decision variables
    if NonNegativeDirection:
        x = model.addVars(dimension, lb=0, ub=1.0, name="x")
    else:
        x = model.addVars(dimension, lb=-1.0, ub=1.0, name="x")
    c = model.addVar(lb=0.0, ub=1.0, name="c")


    # Beam constraints: +/- x^T u_e <= c,
    for i in range(len(vecs)):
        if dimension ==3:
            expr_pos = vecs[i,0]*x[0] + vecs[i,1]*x[1] + vecs[i,2]*x[2]
        else:
            expr_pos = vecs[i,0]*x[0] + vecs[i,1]*x[1]
        model.addConstr(expr_pos <= c, name=f"beam_pos_{i}")
        model.addConstr(-expr_pos <= c, name=f"beam_neg_{i}")

    # Plane constraints: ((x^T n_ef )^2 + c^2 >= 1 
    # and also prepare norm constraint on disturbance vector
    if dimension==3:
        for idx,(i,j,n) in enumerate(normals):
            qexpr = (n[0]*x[0] + n[1]*x[1] + n[2]*x[2]) * (n[0]*x[0] + n[1]*x[1] + n[2]*x[2]) + c*c
            model.addQConstr(qexpr >= 1.0, name=f"plane_{i}_{j}")
            
        qnorm = x[0]*x[0] + x[1]*x[1] + x[2]*x[2]

    else:
        # There are no plane constraints in 2D...
  
        qnorm = x[0]*x[0] + x[1]*x[1]

    # Unit-norm constraint: x^T x == 1
    model.addQConstr(qnorm == 1.0, name="unit_norm")

    # Objective
    model.setObjective(c, GRB.MINIMIZE)

    # Warm start 
    if warmstart_samples and warmstart_samples > 0:
        ws = sample_warm_start(vecs, normals, n_samples=warmstart_samples)
        if ws is not None:
            x_ws, c_ws = ws
            # set start
            for k in range(dimension):
                x[k].start = float(x_ws[k])
            c.start = float(c_ws)
            if verbose:
                print("Warm start provided from sampling: c =", c_ws)
        else:
            if verbose:
                print("No feasible warm start found by sampling (will let Gurobi search).")

    # Optimize
    model.optimize()

    status = model.Status
    result = {'status': status}

    if status == GRB.OPTIMAL or status == GRB.TIME_LIMIT or status == GRB.SUBOPTIMAL:
        x_opt = np.array([x[i].X for i in range(dimension)], dtype=float)
        c_opt = float(c.X)

        c_opt = float(np.clip(c_opt, 0.0, 1.0))
        t_rad = np.arccos(np.clip(c_opt, -1.0, 1.0))
        t_deg = np.degrees(t_rad)

        # find nearest beam (abs dot) and nearest plane (smallest |n^T x|)
        abs_dots = np.arccos(np.abs(vecs @ x_opt) if m_beams>0 else np.array([]))
        nearest_beam_idx = int(np.argmax(abs_dots)) if m_beams>0 else None
        nearest_beam_val = float(abs_dots[nearest_beam_idx]) if nearest_beam_idx is not None else None

        nearest_plane_idx = None
        nearest_plane_val = None

        # "nearest plane" in the sense of smallest angle out of plane -> smallest |n^T x|
        if dimension==3:
            plane_vals = np.arcsin([abs(float(n @ x_opt)) for (_,_,n) in normals])
            print(plane_vals)

            nearest_plane_idx = int(np.argmin(plane_vals))
            nearest_plane_val = float(plane_vals[nearest_plane_idx])

        result.update({
            'x': x_opt,
            'c': c_opt,
            't_rad': t_rad,
            't_deg': t_deg,
            'nearest_beam_idx': nearest_beam_idx,
            'nearest_beam_val': nearest_beam_val,
            'nearest_plane_idx': nearest_plane_idx,
            'nearest_plane_val': nearest_plane_val,
            'num_beams': m_beams,
            'num_planes': len(normals)
        })
        
    else:
        if verbose:
            print("Gurobi did not return a usable solution. Status:", status)
        result.update({'x': None, 'c': None})

    return result


# Parameters

## Your Data and Parameters
For 3D groundstructures, set dimension=3, and name the columns in your csv "x1;y1;z1;x2;y2;z2", where (x1,y1,z1) is one endpoint and (x2,y2,z2) is the other endpoint of a potential member. 

For 3D groundstructures, set dimension=2, and name the columns in your csv "x1;y1;z1;x2;y2;z2", where (x1,y1,z1) is one endpoint and (x2,y2,z2) is the other endpoint of a potential member.

Set SingleDirection=True, if you want to obtain a single direction of the disturbance force, that takes into account ALL existing edges, regardless at which node. I SingleDirection=False, a separate direction for every node will be returned, only taking into account the adjacent edges.

Set NonNegativeDirection=True, if you want the direction to be in the first quadrant/octant, i.e., if you want every entry to be non-negative. If NonNegativeDirection=False, the direction is not restricted to a quadrant/octant.

The value eps describes the selected accuracy. In particular, values that are less than eps are considered to be zero. This is in particular relevant when it comes to detecting parallel members and planes.



In [ ]:
file='data/2_cantilever/9x6/cantileverLarge_ground.csv'
dimension=2
SingleDirection=True
NonNegativeDirection=True
eps=1e-5

# Run

In [ ]:
beams, Normals, nodes= read_csv(file, dimension, SingleDirection,eps) 

if SingleDirection:
    solution = []
    obj = []
    sol = solve_disturbance_direction(dimension,beams,Normals,eps,1000,2000,True)
    print("\nSolution summary:")
    for k,v in sol.items():
        print(k, ":", v)
        if k=="x":
            solution.append(v)
        if k=="c":
            obj.append(v)

    print("Obj:",obj[0]," vector:",solution[0])

    for i in range(dimension):
        print("d_",i,"=",solution[0][i],sep="")

else:
    allSolutions=[]
    allObjectives=[]
    nodeNumbers=[]
    for node in nodes:
        nodeNumbers.append(node)
        adjacentBeams=np.asarray(beams[node], dtype=float)
        if dimension==3:
            sol = solve_disturbance_direction(dimension,adjacentBeams,Normals[node],1e-8,1000,2000,False)
        else:
            sol = solve_disturbance_direction(dimension,adjacentBeams,[],1e-8,1000,2000,False)

        print("\nSolution summary:")
        for k,v in sol.items():
            print(k, ":", v)
            if k=="x":
                allSolutions.append(v)
            if k=="c":
                allObjectives.append(v)
        print("Solution for node ",node)
        print("Obj:",allObjectives[-1]," vector:",allSolutions[-1])
        for i in range(dimension):
            print("d_",i,"=",allSolutions[-1][i],sep="")
    for i in range(len(allSolutions)):
        print("Disturbance vector at node",nodeNumbers[i],":",allSolutions[i],", with objective value",allObjectives[i])
        